In [46]:
from muscriptor import TranscriptionModel
import torchaudio
from muscriptor.tokenizer import note_event2midi, note2note_event
import subprocess
import torch


soundfont_path = "/home/simon/musescore/MuseScore_General.sf2"


model = TranscriptionModel.load_model(
    # weights_path="/home/simon/muscriptor/muscriptor/muscriptor_weights_01684fbb_350.safetensors"
    weights_path="/home/simon/muscriptor/muscriptor/muscriptor_weights_0ac4ce03_900.safetensors"
    # weights_path="/home/simon/muscriptor/muscriptor/muscriptor_weights_e84904c4_700.safetensors"
    # weights_path="/home/simon/muscriptor/muscriptor/muscriptor_weights_8f59580c_750.safetensors"
)

/home/simon/micromamba/envs/muscriptor1/lib/python3.10/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (512) may be set too high. Or, the value for `n_freqs` (1025) may be set too low.
  warnings.warn(


In [51]:
# wav_path = "/home/simon/audio/filling_the_void.wav"
wav_path = "/home/simon/audio/ITSBEENSOLONG.mp3"
wav_path = "/home/simon/audio/coltrane_india_3min.mp3"
wav_path = "/home/simon/audio/lofi_music.mp3"
wav_path = "/home/simon/audio/bad_bunny.mp3"
wav_path = "/home/simon/audio/bruno_mars_risk_it_all.mp3"
wav_path = "/lustre/scwpod02/client/kyutai-interns/datasets/disco_12M_big_01_25/data-5/b88/b881d86d643465683880669caa33052e10a88053.wav"
wav_path = "/home/simon/audio/nir_felder.mp3"


notes = model.transcribe(
    wav_path,
    output_midi="filling_the_void.mid",
    return_notes=True,
    # instrument_group='5 6 8 17 36',
    # instrument_group='0 7 24',
    # instrument_group='4 9 17 23 36',
    instrument_group="0 5 12 36",
    # instrument_group='8',
    dataset_name="0",
    cfg_coef=1.0,
)

print(f"Transcribed {len(notes)} notes")
notes[:5]  # preview first few

[muscriptor] step    0: token=[[1134], [1161], [1161], [1161], [1161], [1161], [1135], [1135], [1135], [1135], [1135], [1161], [1161], [1135], [1135], [1135]]
[muscriptor] step    1: token=[[100], [1044], [1049], [1049], [1049], [1052], [1044], [1044], [1049], [1049], [1063], [1068], [1078], [1049], [1049], [1056]]
[muscriptor] step    2: token=[[1133], [1059], [1056], [1056], [1056], [1056], [1051], [1051], [1056], [1056], [1075], [1078], [1134], [1056], [1056], [1059]]
[muscriptor] step    3: token=[[1309], [1070], [1059], [1059], [1059], [1059], [1056], [1056], [1059], [1059], [1161], [1168], [30], [1059], [1061], [1061]]
[muscriptor] step    4: token=[[113], [1134], [1064], [1064], [1064], [1063], [1059], [1059], [1064], [1064], [1063], [1044], [1133], [1064], [1064], [1064]]
[muscriptor] step    5: token=[[1309], [15], [1068], [1068], [1068], [1068], [1063], [1063], [1068], [1068], [1075], [1134], [1301], [1068], [1068], [1068]]
[muscriptor] step    6: token=[[127], [1132], [1134]

[Note(is_drum=True, program=128, onset=0.97, offset=0.98, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.1, offset=1.11, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.24, offset=1.25, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.38, offset=1.39, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.52, offset=1.53, pitch=44, velocity=1)]

In [52]:
notes

[Note(is_drum=True, program=128, onset=0.97, offset=0.98, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.1, offset=1.11, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.24, offset=1.25, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.38, offset=1.39, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.52, offset=1.53, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.66, offset=1.67, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.8, offset=1.81, pitch=44, velocity=1),
 Note(is_drum=True, program=128, onset=1.94, offset=1.95, pitch=44, velocity=1),
 Note(is_drum=False, program=26, onset=2.03, offset=2.2, pitch=45, velocity=1),
 Note(is_drum=False, program=26, onset=2.03, offset=2.2, pitch=52, velocity=1),
 Note(is_drum=False, program=26, onset=2.03, offset=2.2, pitch=55, velocity=1),
 Note(is_drum=False, program=26, onset=2.03, offset=2.2, pitch=60, velocity=1),
 Note(is_drum=False, program=26, o

In [53]:
wav_path = "/home/simon/audio/filling_the_void.wav"
wav, sr = torchaudio.load(wav_path)
# load and synthesize .npy
note_event2midi(note2note_event(notes), "/home/simon/audio/garbage/output.mid")
subprocess.run(
    [
        "fluidsynth",
        "-ni",
        "-F",
        "/home/simon/audio/garbage/output4.wav",
        "-r",
        str(sr),
        soundfont_path,
        "/home/simon/audio/garbage/output.mid",
    ]
)
wav_trans, sr_trans = torchaudio.load("/home/simon/audio/garbage/output4.wav")

min_len = min(wav.shape[-1], wav_trans.shape[-1])
mixed_all = torch.zeros((2, min_len))
mixed_all[0] = wav.mean(dim=0)[:min_len] * 0.2
mixed_all[1, :min_len] = wav_trans[0, :min_len]

Failed to create secure directory (/run/user/1014/pulse): No such file or directory
fluidsynth: warning: SDL3 not initialized, SDL3 audio driver won't be usable. Have you called SDL_Init(SDL_INIT_AUDIO) ?


FluidSynth runtime version 2.5.4
Copyright (C) 2000-2026 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '/home/simon/audio/garbage/output4.wav'..


# debug

In [ ]:
import math
import torch
from muscriptor.transcription_model import _SAMPLE_RATE, _SEGMENT_DURATION

# 1. Check audio loading
wav_path = "/home/simon/audio/filling_the_void.wav"
wav = model._load_wav(wav_path, None)
print(f"wav: shape={wav.shape}, duration={wav.shape[-1] / _SAMPLE_RATE:.1f}s")

# 2. Check chunking
segment_samples = int(_SEGMENT_DURATION * _SAMPLE_RATE)
num_chunks = math.ceil(wav.shape[-1] / segment_samples)
print(f"Chunks: {num_chunks} × {_SEGMENT_DURATION}s")

# 3. Generate tokens for just the first chunk
chunk0 = wav[:, segment_samples * 11 : segment_samples * 12]


conditions = model._build_conditions(
    chunk0, instrument_group="5 6 8 17 36", dataset_name="0"
)
max_gen_len = model._tokenizer.get_generation_length(_SEGMENT_DURATION)
eos_id = model._tokenizer.tokenizer.eos_id
print(f"max_gen_len={max_gen_len}, eos_id={eos_id}")

with torch.no_grad():
    tokens = model._model.generate(
        conditions=conditions,
        max_gen_len=max_gen_len,
        use_sampling=False,
        cfg_coef=2.0,
        early_stop_on_token=eos_id,
    )
print(f"tokens shape: {tokens.shape}")
print(f"first 30 tokens: {tokens[0, 0, :30].tolist()}")

# 4. Decode that one chunk
model._tokenizer.error_counter.clear()
notes_dict, _ = model._tokenizer.decode(
    tokens[0], seek_time=0.0, segment_duration=_SEGMENT_DURATION
)
print(f"notes from chunk 0: {len(notes_dict['notes'])}")
print(f"tokenizer errors: {dict(model._tokenizer.error_counter)}")

wav: shape=torch.Size([1, 3485715]), duration=217.9s
Chunks: 44 × 5.0s
max_gen_len=1920, eos_id=1
[muscriptor] instrument_group tokens: tensor([[ 6,  7,  9, 18, 37],
        [ 0,  0,  0,  0,  0]], device='cuda:0')
[muscriptor] dataset_name tokens:     tensor([[1],
        [0]], device='cuda:0')
[muscriptor] CFG logits top5: ids=[1187, 1134, 1161, 1168, 1164]  vals=[13.703125, 13.625, 13.2890625, 11.203125, 10.8984375]
[muscriptor] step    0: token=[[1187]]
[muscriptor] CFG logits top5: ids=[1073, 1061, 1064, 1068, 1071]  vals=[8.1640625, 7.625, 5.46875, 3.75390625, 3.328125]
[muscriptor] step    1: token=[[1073]]
[muscriptor] CFG logits top5: ids=[1134, 1187, 1076, 1073, 1085]  vals=[16.765625, 10.453125, 9.0703125, 7.1796875, 6.640625]
[muscriptor] step    2: token=[[1134]]
[muscriptor] CFG logits top5: ids=[27, 26, 28, 29, 25]  vals=[9.28125, 8.9375, 8.890625, 8.421875, 7.984375]
[muscriptor] step    3: token=[[27]]
[muscriptor] CFG logits top5: ids=[1133, 1168, 1132, 1161, 1164]  va

In [3]:
tokens[..., :100]

tensor([[[1187, 1073, 1134,   27, 1133, 1303,   74, 1161, 1049, 1168, 1049,
          1187, 1132, 1073, 1133, 1301,   98, 1161, 1132, 1049, 1133, 1056,
          1168, 1132, 1049, 1133, 1044,  122, 1161, 1132, 1056, 1168, 1044,
          1187, 1133, 1068, 1303,  146, 1168, 1037, 1187, 1132, 1068, 1133,
          1064, 1301,  170, 1168, 1132, 1037, 1187, 1064, 1133, 1301,  194,
          1168, 1042, 1303, 1310,  218, 1132, 1042, 1133, 1044, 1187, 1068,
          1301, 1308,  242, 1161, 1047, 1168, 1132, 1044, 1133, 1047, 1187,
          1132, 1068, 1133, 1066, 1311,  266, 1161, 1069, 1168, 1132, 1047,
          1133, 1047, 1187, 1132, 1066, 1133, 1066, 1301,  290, 1161, 1132,
          1047]]], device='cuda:0')

In [4]:
notes_dict

{'seek_time': 0.0,
 'duration_sec': 5.0,
 'notes': [Note(is_drum=False, program=52, onset=0, offset=0.71, pitch=69, velocity=1),
  Note(is_drum=True, program=128, onset=0.24, offset=0.25, pitch=38, velocity=1),
  Note(is_drum=False, program=26, onset=0.71, offset=0.95, pitch=45, velocity=1),
  Note(is_drum=False, program=33, onset=0.71, offset=0.95, pitch=45, velocity=1),
  Note(is_drum=True, program=128, onset=0.71, offset=0.72, pitch=36, velocity=1),
  Note(is_drum=False, program=26, onset=0.95, offset=1.19, pitch=52, velocity=1),
  Note(is_drum=False, program=33, onset=0.95, offset=1.19, pitch=40, velocity=1),
  Note(is_drum=False, program=52, onset=1.19, offset=1.43, pitch=64, velocity=1),
  Note(is_drum=True, program=128, onset=1.19, offset=1.2, pitch=38, velocity=1),
  Note(is_drum=False, program=33, onset=1.43, offset=1.67, pitch=33, velocity=1),
  Note(is_drum=False, program=52, onset=1.43, offset=1.67, pitch=60, velocity=1),
  Note(is_drum=True, program=128, onset=1.43, offset